In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm
import re
from statsmodels.graphics.tsaplots import plot_acf
import numpy as np

In [ ]:
companies = ["Apple", "Exxon", "JPM", "PFE"]

for ticker in companies:
    # ---------- 1.  read price data ----------
    histfile = {
        "Apple": "AAPL_historical_data.csv",
        "Exxon": "XOM_historical_data.csv",
        "JPM":   "JPM_historical_data.csv",
        "PFE":   "PFE_historical_data.csv",
    }[ticker]
    df = pd.read_csv(histfile)

    # ---------- 2.  models that ranked in the top-5 on AIC ----------
    top5_path = Path(f"result_table_{ticker}/top5.csv")
    top5 = pd.read_csv(top5_path, index_col=0)        # model names live in the index
    model_names = top5.index.tolist()                 # <-- crucial change

    # ---------- 3.  pre-compute residuals for the post-2024 period ----------
    split_date = "2024-01-02"
    df["date"] = pd.to_datetime(df["Date"])
    df["Log_Returns"] = np.log(df["Close"] / df["Close"].shift(1))
    df.dropna(subset=["Log_Returns"], inplace=True)

    mean_ret = df["Log_Returns"].mean()
    df["residuals"] = (df["Log_Returns"] - mean_ret) * 100
    test_df = df[df["date"] >= split_date]

    # ---------- 4.  loop over the five winning volatility models ----------
    for vol in model_names:
        # break the model string into pieces
        prefix_and_prop, polarity = vol.rsplit("+", 1)
        base_model, prop_tuple = prefix_and_prop.rsplit("+", 1)
        vol_filename = f"{base_model}_volatility-{prop_tuple}-{polarity}.csv"
        vol_path = Path(f"volat_table/{ticker}/{vol_filename}")

        # ad-hoc special case stays as-is
        if vol == "garch+('count','Mean')+PolarityRob":
            df_vols = pd.read_csv(
                r"volat_table\Exxon\garch_volatility-prop-PolarityRob.csv",
                index_col=0
            )
        else:
            df_vols = pd.read_csv(vol_path, index_col=0)

        df_vols["date"] = pd.to_datetime(df_vols["date"])

        # ---------- 5.  merge & analyse ----------
        merged = test_df[["date", "residuals"]].merge(
            df_vols[["date", "realized_vol", "relative_volat"]],
            on="date",
            how="inner"
        )
        white_noise = merged["residuals"] / merged["relative_volat"]

        # ---------- 6.  plotting ----------
        fig, axs = plt.subplots(2, 2, figsize=(14, 10))

        # 6-a realised vs relative vol
        ax0 = axs[0, 0]
        ax0.plot(merged["date"], merged["realized_vol"], label="Realized Vol", lw=1)
        ax0.plot(merged["date"], merged["relative_volat"], label="Relative Vol", lw=1)
        ax0.set_title("Realized vs Relative Volatility")
        ax0.set_xlabel("Date")
        ax0.set_ylabel("Volatility")
        ax0.legend()
        ax0.grid(alpha=0.3)

        # 6-b standardised residuals
        ax1 = axs[0, 1]
        ax1.plot(white_noise, lw=1)
        ax1.axhline(0, ls="--", color="gray")
        ax1.set_title("Standardised Residuals")
        ax1.set_xlabel("Observation")
        ax1.set_ylabel("Std. Residual")
        ax1.grid(alpha=0.3)

        # 6-c ACF
        ax2 = axs[1, 0]
        plot_acf(white_noise, lags=40, alpha=0.05, ax=ax2)
        ax2.set_title("ACF of Standardised Residuals")
        ax2.set_xlabel("Lag")
        ax2.set_ylabel("ACF")

        # 6-d QQ-plot
        ax3 = axs[1, 1]
        sm.qqplot(white_noise, line="45", fit=True, ax=ax3)
        ax3.set_title("QQ-Plot of Standardised Residuals")
        ax3.grid(alpha=0.3)

        fig.suptitle(f"{ticker} – {vol}", fontsize=16)
        plt.tight_layout()

        out_png = Path(f"plots/{ticker}/{vol}.png")
        out_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

        print(f"Saved combined figure to {out_png}")

        print(vol)
        txt = Path(f"white_tests_{ticker}/{vol}.txt")

        # ── 2. helper: try a regex; fill NaN if it isn’t found ──────────────────────────
        def grab(text, pattern, n_groups=2):
            """
            Return a tuple of floats (stat, p) or NaNs if the pattern is missing.
            n_groups=1 → only 'stat' is expected (no p-value in the file).
            """
            m = re.search(pattern, text, flags=re.I)
            if not m:
                return (pd.NA,) * n_groups
            return tuple(float(x) for x in m.groups())

        # ── 3. loop over every file ─────────────────────────────────────────────────────
        rows = []
        FLOAT = r"[-+]?\d*\.\d+|\d+"          # generic float / int / sci-notation

        txt_str = txt.read_text(encoding="utf-8", errors="ignore")

        row = {
            "file": txt.stem,                                 # filename without .txt
            # the descriptor on the very first line, e.g. ('ushape', 'WeightedMean')
            "descriptor": re.search(r"==\s*\((.*?)\)\+\[", txt_str).group(1)
                        if re.search(r"==\s*\((.*?)\)\+\[", txt_str) else pd.NA,
        }

        # ── Normality tests ────────────────────────────────────────────────────────
        row["JB_stat"], row["JB_p"]       = grab(txt_str,
            fr"Jarque[- ]Bera.*statistic=({FLOAT}).*p-value=({FLOAT})")
        row["SW_stat"], row["SW_p"]       = grab(txt_str,
            fr"Shapiro[- ]Wilk.*statistic=({FLOAT}).*p-value=({FLOAT})")
        row["AD_stat"],                   = grab(txt_str,
            fr"Anderson[- ]Darling.*statistic=({FLOAT})", n_groups=1)

        # ── Autocorrelation (Ljung-Box) ────────────────────────────────────────────
        row["LB_stat"], row["LB_p"]       = grab(txt_str,
            fr"Ljung[- ]Box.*?\n.*?({FLOAT})\s+({FLOAT})")

        # ── Stationarity ───────────────────────────────────────────────────────────
        row["ADF_stat"], row["ADF_p"]     = grab(txt_str,
            fr"ADF test.*statistic=({FLOAT}).*p-value=({FLOAT})")

        # ── Heteroscedasticity ─────────────────────────────────────────────────────
        row["ARCH_stat"], row["ARCH_p"]   = grab(txt_str,
            fr"Engle.*ARCH.*statistic=({FLOAT}).*p-value=({FLOAT})")
        row["BP_stat"], row["BP_p"]       = grab(txt_str,
            fr"Breusch[- ]Pagan.*statistic=({FLOAT}).*p-value=({FLOAT})")

        # ── Group-wise homoscedasticity ────────────────────────────────────────────
        row["Levene_stat"], row["Levene_p"] = grab(txt_str,
            fr"Levene test.*statistic=({FLOAT}).*p-value=({FLOAT})")
        row["Bartlett_stat"], row["Bartlett_p"] = grab(txt_str,
            fr"Bartlett test.*statistic=({FLOAT}).*p-value=({FLOAT})")

        rows.append(row)

    # ── 4. build the table & save ──────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    df.to_csv("summary_white_noise_stats.csv",sep=";", index=False)

    print("Wrote summary_white_noise_stats.csv with", len(df), "rows")
